# sillyrl: loss-configuration sweep on Colab

Trains `base`, `mc`, `td`, `mc_a4` and `td_a4` (see `maze_consistency/experiments.py`) on the canonical maze and
tracks their error against the exact test set during training.

1. **Runtime → Change runtime type → GPU** (optional, much faster than CPU).
2. Set `REPO` and `BRANCH` below. For a **private** repo, add a Colab secret named `GITHUB_TOKEN`
   (key icon in the left bar) holding a GitHub token with read access, and enable notebook access.
3. **Runtime → Run all.** If an import error already happened in this session, first
   **Runtime → Restart session**, so the upgraded flax is the one that gets imported.

Runs are written to Google Drive (`RUNS_DIR`), so if the session disconnects, run all again: finished
runs are skipped and the sweep continues where it stopped.

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "colab-sweep"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}

In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks) and the exact test set from data/canonical/maze.txt. Deterministic.
!python run.py dataset | tail -4
!python run.py testset | tail -1

In [ ]:
CONFIGS = "base,mc,td,mc_a4,td_a4"  #@param {type:"string"}
SEEDS = "0"  #@param {type:"string"}
STEPS = 3000  #@param {type:"integer"}
BATCH = 32  #@param {type:"integer"}
EVAL_EVERY = 250  #@param {type:"integer"}
EVAL_PER_SETTING = 50  #@param {type:"integer"}

from maze_consistency.experiments import SweepConfig, run_sweep
sc = SweepConfig(configs=tuple(c.strip() for c in CONFIGS.split(",")),
                 seeds=tuple(int(s) for s in SEEDS.split(",")),
                 steps=STEPS, batch=BATCH, eval_every=EVAL_EVERY, eval_per_setting=EVAL_PER_SETTING)
print(sc)
run_sweep(sc)

In [ ]:
# Test-metric curves over training, one line per config.
from IPython.display import Image, display
from maze_consistency.experiments import plot_sweep
plot_sweep(sc.prefix)
display(Image(os.path.join(os.environ["RUNS_DIR"], sc.prefix, "curves.png")))

In [ ]:
# Final exact-test metrics per config (mean over seeds). All values are in nats; lower is better.
import pandas as pd
from maze_consistency.experiments import load_sweep
rows = [dict(config=cfg, **hist[-1]) for cfg, hists in load_sweep(sc.prefix).items() for hist in hists]
final = pd.DataFrame(rows).drop(columns="step").groupby("config").mean()
cols = [c for c in final.columns if c.split("/")[1] in ("NOR", "bin 10", "bin 11", "best far")
        and c.split("/")[0] in ("act_kl", "value_kl")]
final[cols].round(4)